# Relativistic markers

`boris_push(..., relativistic=True)` pushes markers that move close to the speed of light: MeV electrons, runaway or beam particles. It advances the proper velocity $\mathbf u = \gamma \mathbf v$ (the momentum over the rest mass) instead of the velocity, with the relativistic Boris scheme:

- half an electric kick, $\mathbf u^- = \mathbf u + \frac{q}{m}\mathbf E \frac{\Delta t}{2}$;
- a rotation about $\frac{q}{m}\mathbf B \frac{\Delta t}{2\gamma^-}$, with $\gamma^- = \sqrt{1 + |\mathbf u^-|^2/c^2}$;
- the other half kick, then the drift $\mathbf x \mathrel{+}= \Delta t\, \mathbf u / \gamma$.

$|\mathbf u|$ is unbounded while $|\mathbf v| = |\mathbf u|/\gamma < c$, so no marker gets faster than light, and for $|\mathbf u| \ll c$ the step is the classical Boris step of [Pushing markers](/mpiparticles/tutorials/pushing-markers/). A store pushed this way holds $\mathbf u$ in `store.velocities`: `proper_velocity` makes it from velocities, `coordinate_velocity` turns it back, and `lorentz_factor` and `kinetic_energy` give $\gamma$ and $(\gamma - 1) m c^2$.

This tutorial gyrates a 1 MeV electron, accelerates one in a uniform electric field and follows the relativistic E×B drift.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mpiparticles import (
    SPEED_OF_LIGHT,
    ParticleStore,
    boris_push,
    boris_velocity,
    coordinate_velocity,
    kinetic_energy,
    lorentz_factor,
    proper_velocity,
)

E_CHARGE = 1.602176634e-19  # [C]
E_MASS = 9.1093837015e-31  # [kg]
ELECTRON = -E_CHARGE / E_MASS  # charge over rest mass [C/kg]
MEV = 1e6 * E_CHARGE  # [J]

## Gyration

A 1 MeV electron has $\gamma = 1 + E_\mathrm{kin}/(m c^2) \approx 2.96$ and moves at $0.94c$; the classical formula $\sqrt{2E_\mathrm{kin}/m}$ would give almost $2c$. In a magnetic field $B = 0.1$ T it gyrates at $\omega_c/\gamma$ on the radius $\gamma v / \omega_c$. Pushed classically with the same velocity, it would gyrate $\gamma$ times faster on a circle $\gamma$ times smaller. The step is $\omega_c \Delta t = 0.05$; both stores are pushed for one relativistic gyration period.

In [ ]:
b = np.array([0.0, 0.0, 0.1])
omega_c = abs(ELECTRON) * b[2]
gamma = 1 + MEV / (E_MASS * SPEED_OF_LIGHT**2)
v = SPEED_OF_LIGHT * np.sqrt(1 - 1 / gamma**2)
print(f"gamma = {gamma:.3f}, v = {v / SPEED_OF_LIGHT:.3f} c")

dt = 0.05 / omega_c
steps = int(2 * np.pi * gamma / 0.05)
orbits = {}
for relativistic in (False, True):
    velocity = np.array([[v], [0.0], [0.0]])
    store = ParticleStore(ndim=2, nvel=3)
    store.add(np.zeros((2, 1)), proper_velocity(velocity) if relativistic else velocity)
    orbit = []
    for _ in range(steps):
        boris_push(store, [0.0, 0.0, 0.0], b, dt, ELECTRON, relativistic=relativistic)
        orbit.append(store.positions[:, 0].copy())
    orbits[relativistic] = np.array(orbit) * 1e3  # mm
    u = store.velocities[:, :1]
    speed = np.linalg.norm(coordinate_velocity(u) if relativistic else u)
    print(
        f"relativistic={relativistic}: radius {np.ptp(orbits[relativistic][:, 0]) / 2:.2f} mm, "
        f"speed after the push {speed / SPEED_OF_LIGHT:.6f} c"
    )
print(f"expected radius gamma v / omega_c = {gamma * v / omega_c * 1e3:.2f} mm")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(*orbits[True].T, label="relativistic")
ax.plot(*orbits[False].T, label="classical")
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title("A 1 MeV electron in 0.1 T, one period")
ax.legend()
plt.show()

The rotation only turns $\mathbf u$, so $|\mathbf u|$, $\gamma$ and the energy are kept exactly; the phase lags by the Boris angle $2\arctan(\omega_c\Delta t/(2\gamma))$ per step instead of $\omega_c\Delta t/\gamma$.

## Acceleration in an electric field

An electron starting from rest in $E = 1$ MV/m gains proper velocity at the constant rate $a = |q|E/m$, so $u = a t$ and $v = a t / \sqrt{1 + (a t/c)^2}$: the speed approaches $c$ but never reaches it (hyperbolic motion). The classical push gains velocity at the same rate and passes the speed of light after 1.7 ns. `boris_velocity` with half a negative step starts the leapfrog: the velocities are then known at the half steps.

In [ ]:
e = np.array([1e6, 0.0, 0.0])
a = abs(ELECTRON) * e[0]
dt, steps = 1e-11, 1000
speeds = {}
for relativistic in (False, True):
    store = ParticleStore(ndim=1, nvel=3)
    store.add(np.zeros((1, 1)), np.zeros((3, 1)))
    boris_velocity(
        store.velocities[:, :1],
        e,
        [0.0, 0.0, 0.0],
        -0.5 * dt,
        ELECTRON,
        relativistic=relativistic,
    )
    speed = []
    for _ in range(steps):
        boris_push(store, e, [0.0, 0.0, 0.0], dt, ELECTRON, relativistic=relativistic)
        u = store.velocities[:, :1]
        speed.append(-(coordinate_velocity(u) if relativistic else u)[0, 0])
    speeds[relativistic] = np.array(speed) / SPEED_OF_LIGHT

t = (np.arange(steps) + 0.5) * dt  # the velocities are at the half steps
exact = a * t / np.sqrt(1 + (a * t / SPEED_OF_LIGHT) ** 2) / SPEED_OF_LIGHT
print(
    f"largest error of the relativistic speed: {np.abs(speeds[True] - exact).max():.1e} c"
)
print(
    f"kinetic energy after {steps * dt * 1e9:.0f} ns: {kinetic_energy(store.velocities[:, :1], E_MASS)[0] / MEV:.2f} MeV"
)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t * 1e9, speeds[False], label="classical")
ax.plot(t * 1e9, speeds[True], label="relativistic")
ax.plot(t * 1e9, exact, "k:", label="exact")
ax.axhline(1.0, color="gray", lw=0.8)
ax.set_ylim(0, 2)
ax.set_xlabel("t [ns]")
ax.set_ylabel("v / c")
ax.set_title("An electron from rest in 1 MV/m")
ax.legend()
plt.show()

The kick changes $\mathbf u$ by exactly $\frac{q}{m}\mathbf E\Delta t$, so the proper velocity is exact at every half step and the speed only carries round-off; the position is the midpoint rule of $\int \mathbf v\,dt$, second order in $\Delta t$.

## The relativistic E×B drift

In crossed fields with $E < cB$ the guiding centre drifts with $\mathbf E\times\mathbf B/B^2$, here $E = 0.6\,cB$: the drift is $0.6c$, and the electron, starting at rest, reaches $(0.6 + 0.6)c/(1 + 0.6^2) = 0.88c$, $\gamma \approx 2.1$, at the top of each arc. The cycloid is stretched along the drift compared to the classical one. (With $E > cB$ there is no frame without electric field, and the electron is accelerated without bound.)

In [ ]:
B = 0.01  # [T]
E = 0.6 * SPEED_OF_LIGHT * B  # [V/m]
omega_c = abs(ELECTRON) * B
dt, steps = 0.05 / omega_c, 1500
store = ParticleStore(ndim=2, nvel=3)
store.add(np.zeros((2, 1)), np.zeros((3, 1)))
path, gammas = [], []
for _ in range(steps):
    boris_push(store, [0.0, E, 0.0], [0.0, 0.0, B], dt, ELECTRON, relativistic=True)
    path.append(store.positions[:, 0].copy())
    gammas.append(lorentz_factor(store.velocities[:, :1])[0])
path = np.array(path)
print(
    f"mean drift {path[-1, 0] / (steps * dt) / SPEED_OF_LIGHT:.3f} c (E/B = 0.6 c), "
    f"largest gamma {max(gammas):.2f}"
)

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.plot(path[:, 0], path[:, 1])
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("E = 0.6 cB: the relativistic cycloid of an electron")
plt.show()

## More markers, plain arrays and other units

The relativistic push takes everything the classical one does: 1D3V to 3D3V stores, a charge over rest mass and a time step per marker, fields per marker or the same everywhere, CuPy arrays (CUDA kernels). `boris_velocity(..., relativistic=True)` is the update of the proper velocities alone, on a plain `(3, N)` array. `speed_of_light` sets $c$ in other units, e.g. `1.0` in normalised units:

In [ ]:
rng = np.random.default_rng(0)
u = rng.normal(size=(3, 5))  # proper velocities in units of c
boris_velocity(
    u, [0.0, 0.0, 0.0], [0.0, 0.0, 2.0], 0.1, 1.0, relativistic=True, speed_of_light=1.0
)
print("gamma:", np.round(lorentz_factor(u, speed_of_light=1.0), 3))
print(
    "v / c:",
    np.round(np.linalg.norm(coordinate_velocity(u, speed_of_light=1.0), axis=0), 3),
)

The moments of a relativistic store (`velocity_moments`, `phase_space_density`) are moments of $\mathbf u$; convert with `coordinate_velocity` first for those of $\mathbf v$.

For codes that fuse the push into their own kernels, the step is a block in every backend, as the classical one: `mpiparticles_relativistic_boris` and `mpiparticles_lorentz_factor` in `mpiparticles/kernels/boris.cuh` (CUDA), `relativistic_boris_update` and `lorentz_factor` in `mpiparticles.kernels.boris` (NumPy and `RELATIVISTIC_BORIS_METAL_HEADER`) and in `mpiparticles.kernels.boris_pyccel` (Pyccel, inlined).